# Projeto 2

## João Victor De Bortoli Prado - 13672071

## Conceito

* Garagem na beira de estrada
* Objetos internos: lâmpada, pneu, ferramentas, gasolina
* Objetos externos: árvores, póstes, cones
* Objeto extra: carro
* Ambiente: estrada, grama, terra, garagem, portão

## Controles

* W/A/S/D - movimentação da câmera
* P     - liga/desliga modo wireframe
* R     - reseta a cena
* ↓/↑   - movimenta carro para frete/trás
* ←/→   - move o volante (do carro) para esquerda/direita
* SPACEBAR - move o volante (do carro) para o centro
* I - enche o pneu
* K - esvazia o pneu
* O - abre o portão
* L - fecha o portão

### Importando bibliotecas necessárias

In [757]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
from numpy import random
from PIL import Image

from shader_s import Shader

### Inicializando janela

In [758]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 700
largura = 700

window = glfw.create_window(largura, altura, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)


### Constroi, compila e "linka" shaders aos programas

In [759]:
ourShader = Shader("vertex_shader.vs", "fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

### Preparando dados e carregando modelos

In [760]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable( GL_BLEND )
glBlendFunc( GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA )
glEnable(GL_LINE_SMOOTH)


global vertices_list
vertices_list = []    
global textures_coord_list
textures_coord_list = []


def load_model_from_file(filename):
    """Loads a Wavefront OBJ file. """
    objects = {}
    vertices = []
    texture_coords = []
    faces = []

    material = None

    # abre o arquivo obj para leitura
    for line in open(filename, "r"): ## para cada linha do arquivo .obj
        if line.startswith('#'): continue ## ignora comentarios
        values = line.split() # quebra a linha por espaço
        if not values: continue

        ### recuperando vertices
        if values[0] == 'v':
            vertices.append(values[1:4])

        ### recuperando coordenadas de textura
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])

        ### recuperando faces 
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]
        elif values[0] == 'f':
            face = []
            face_texture = []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)

            faces.append((face, face_texture, material))

    model = {}
    model['vertices'] = vertices
    model['texture'] = texture_coords
    model['faces'] = faces

    return model


def load_texture_from_file(texture_id, img_textura):
    print(texture_id)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR_MIPMAP_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    img = Image.open(img_textura)
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGB", 0, -1)
    #image_data = np.array(list(img.getdata()), np.uint8)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)
    glGenerateMipmap(GL_TEXTURE_2D)
    try:
        from OpenGL.GL.EXT.texture_filter_anisotropic import GL_TEXTURE_MAX_ANISOTROPY_EXT
        glTexParameterf(GL_TEXTURE_2D, GL_TEXTURE_MAX_ANISOTROPY_EXT, 16.0)
    except:
        pass

def load_texture_from_file_skybox(texture_id, img_textura):
    print(texture_id)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR_MIPMAP_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    img = Image.open(img_textura)
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGB", 0, -1)
    #image_data = np.array(list(img.getdata()), np.uint8)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)
    glGenerateMipmap(GL_TEXTURE_2D)
    try:
        from OpenGL.GL.EXT.texture_filter_anisotropic import GL_TEXTURE_MAX_ANISOTROPY_EXT
        glTexParameterf(GL_TEXTURE_2D, GL_TEXTURE_MAX_ANISOTROPY_EXT, 16.0)
    except:
        pass



'''
É possível encontrar, na Internet, modelos .obj cujas faces não sejam triângulos. Nesses casos, precisamos gerar triângulos a partir dos vértices da face.
A função abaixo retorna a sequência de vértices que permite isso. Créditos: Hélio Nogueira Cardoso e Danielle Modesti (SCC0650 - 2024/2).
'''
def circular_sliding_window_of_three(arr):
    if len(arr) == 3:
        return arr
    circular_arr = arr + [arr[0]]
    result = []
    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])
    return result
    
global numberTextures
numberTextures = 0

def load_obj_and_texture(objFile, texturesList):
    modelo = load_model_from_file(objFile)
    
    ### inserindo vertices do modelo no vetor de vertices
    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))
    faces_visited = []
    for face in modelo['faces']:
        if face[2] not in faces_visited:
            faces_visited.append(face[2])
        for vertice_id in circular_sliding_window_of_three(face[0]):
            vertices_list.append(modelo['vertices'][vertice_id - 1])
        for texture_id in circular_sliding_window_of_three(face[1]):
            textures_coord_list.append(modelo['texture'][texture_id - 1])
        
    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))
    
    ### carregando textura equivalente e definindo um id (buffer): use um id por textura!
    global numberTextures
    for i in range(len(texturesList)):
        load_texture_from_file(numberTextures,texturesList[i])
        numberTextures += 1
    
    return verticeInicial, verticeFinal - verticeInicial

### Definindo funções de desenho para cada modelo

In [761]:
# carrega rua (modelo e texturas)
verticeInicial_rua, quantosVertices_rua = load_obj_and_texture('objetos/rua/rua.obj', ['objetos/rua/rua.png'])

def desenha_rua(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_rua, quantosVertices_rua) ## renderizando

# carrega poste (modelo e texturas)
verticeInicial_poste, quantosVertices_poste = load_obj_and_texture('objetos/poste/poste.obj', ['objetos/poste/poste.png'])

def desenha_poste(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_poste, quantosVertices_poste) ## renderizando

# carrega carro (modelo e texturas)

verticeInicial_carro, quantosVertices_carro = load_obj_and_texture('objetos/carro/carro.obj', ['objetos/carro/carro.png'])

def desenha_carro(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_carro, quantosVertices_carro) ## renderizando

# carrega pneu (modelo e texturas)

verticeInicial_pneu, quantosVertices_pneu = load_obj_and_texture('objetos/pneu/pneu.obj', ['objetos/pneu/pneu.png'])

def desenha_pneu(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):

    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_pneu, quantosVertices_pneu) ## renderizando

# carrega cone (modelo e texturas)

verticeInicial_cone, quantosVertices_cone = load_obj_and_texture('objetos/cone/cone.obj', ['objetos/cone/cone.png'])

def desenha_cone(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_cone, quantosVertices_cone) ## renderizando

# carrega ferramentas (modelo e texturas)

verticeInicial_ferramentas, quantosVertices_ferramentas = load_obj_and_texture('objetos/ferramentas/ferramentas.obj', ['objetos/ferramentas/ferramentas.png'])

def desenha_ferramentas(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_ferramentas, quantosVertices_ferramentas) ## renderizando

# carrega gasolina (modelo e texturas)

verticeInicial_gasolina, quantosVertices_gasolina = load_obj_and_texture('objetos/gasolina/gasolina.obj', ['objetos/gasolina/gasolina.png'])

def desenha_gasolina(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_gasolina, quantosVertices_gasolina) ## renderizando

# carrega lampada (modelo e texturas)

verticeInicial_lampada, quantosVertices_lampada = load_obj_and_texture('objetos/lampada/lampada.obj', ['objetos/lampada/lampada.png'])

def desenha_lampada(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_lampada, quantosVertices_lampada) ## renderizando

# carrega arvore (modelo e texturas)

verticeInicial_arvore, quantosVertices_arvore = load_obj_and_texture('objetos/arvore/arvore.obj', ['objetos/arvore/arvore.png'])

def desenha_arvore(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)
           
    #define id da textura do modelo
    glBindTexture(GL_TEXTURE_2D, textureId)
    
    # desenha o modelo
    glDrawArrays(GL_TRIANGLES, verticeInicial_arvore, quantosVertices_arvore) ## renderizando

Processando modelo objetos/rua/rua.obj. Vertice inicial: 0
Processando modelo objetos/rua/rua.obj. Vertice final: 90720
0
Processando modelo objetos/poste/poste.obj. Vertice inicial: 90720
Processando modelo objetos/poste/poste.obj. Vertice final: 132366
1
Processando modelo objetos/carro/carro.obj. Vertice inicial: 132366
Processando modelo objetos/carro/carro.obj. Vertice final: 152601
2
Processando modelo objetos/pneu/pneu.obj. Vertice inicial: 152601
Processando modelo objetos/pneu/pneu.obj. Vertice final: 361641
3
Processando modelo objetos/cone/cone.obj. Vertice inicial: 361641
Processando modelo objetos/cone/cone.obj. Vertice final: 369009
4
Processando modelo objetos/ferramentas/ferramentas.obj. Vertice inicial: 369009
Processando modelo objetos/ferramentas/ferramentas.obj. Vertice final: 429639
5
Processando modelo objetos/gasolina/gasolina.obj. Vertice inicial: 429639
Processando modelo objetos/gasolina/gasolina.obj. Vertice final: 484563
6
Processando modelo objetos/lampada/

In [762]:
def cria_skybox(texturas):
    global vertices_list, textures_coord_list

    size = 200.0

    # vértices (36 = 6 faces * 2 triângulos * 3 vértices)
    cube = [
        # frente
        (-size, -size,  size), ( size, -size,  size), ( size,  size,  size),
        (-size, -size,  size), ( size,  size,  size), (-size,  size,  size),

        # direita
        ( size, -size,  size), ( size, -size, -size), ( size,  size, -size),
        ( size, -size,  size), ( size,  size, -size), ( size,  size,  size),

        # trás
        ( size, -size, -size), (-size, -size, -size), (-size,  size, -size),
        ( size, -size, -size), (-size,  size, -size), ( size,  size, -size),

        # esquerda
        (-size, -size, -size), (-size, -size,  size), (-size,  size,  size),
        (-size, -size, -size), (-size,  size,  size), (-size,  size, -size),

        # baixo
        (-size, -size, -size), ( size, -size, -size), ( size, -size,  size),
        (-size, -size, -size), ( size, -size,  size), (-size, -size,  size),

        # cima
        (-size,  size,  size), ( size,  size,  size), ( size,  size, -size),
        (-size,  size,  size), ( size,  size, -size), (-size,  size, -size),
    ]

    # UV padrão (cada face usa imagem inteira)
    uv = [
        (0,0), (1,0), (1,1),
        (0,0), (1,1), (0,1),
    ] * 6

    inicio = len(vertices_list)

    for v in cube:
        vertices_list.append(v)

    for t in uv:
        textures_coord_list.append(t)

    quantidade = len(cube)

    # carregar texturas (6 imagens)
    global numberTextures
    textura_ids = []

    for tex in texturas:
        load_texture_from_file_skybox(numberTextures, tex)
        textura_ids.append(numberTextures)
        numberTextures += 1

    return inicio, quantidade, textura_ids

def desenha_skybox(inicio, textura_ids):
    glDisable(GL_CULL_FACE)  # ver por dentro

    for i in range(6):
        glBindTexture(GL_TEXTURE_2D, textura_ids[i])
        glDrawArrays(GL_TRIANGLES, inicio + i*6, 6)

    glEnable(GL_CULL_FACE)


skybox_texturas = [
    "objetos/skybox/cloudy/bluecloud_rt.jpg", # trás
    "objetos/skybox/cloudy/bluecloud_ft.jpg", # direita
    "objetos/skybox/cloudy/bluecloud_lf.jpg", # frente
    "objetos/skybox/cloudy/bluecloud_bk.jpg", # esquerda
    "objetos/skybox/cloudy/bluecloud_dn.jpg", # baixo
    "objetos/skybox/cloudy/bluecloud_up.jpg" # cima
    ]

skybox_inicio, skybox_qtd, skybox_tex = cria_skybox(skybox_texturas)


9
10
11
12
13
14


In [763]:
def cria_piso(textura, repeat=1):
    global vertices_list, textures_coord_list

    size = 1.0  # tamanho grande

    # vértices (2 triângulos)
    piso = [
    (-size,-0.1,-size), ( size,-0.1, size), ( size,-0.1,-size),
    (-size,-0.1,-size), (-size,-0.1, size), ( size,-0.1, size),
    ]


    # UV padrão
    uv = [
        (0,0), (repeat,0), (repeat,repeat),
        (0,0), (repeat,repeat), (0,repeat),
    ]


    inicio = len(vertices_list)

    for v in piso:
        vertices_list.append(v)

    for t in uv:
        textures_coord_list.append(t)

    quantidade = len(piso)

    # carregar textura
    global numberTextures
    load_texture_from_file(numberTextures, textura)
    textura_id = numberTextures
    numberTextures += 1

    return inicio, quantidade, textura_id

def desenha_piso(inicio, textura_id, tx, ty, tz, sx=1, sy=1, sz=1):

    mat_model = model(
        0,          # angulo
        0,0,1,      # eixo rotação (irrelevante aqui)
        tx, ty, tz, # translação
        sx, sy, sz  # escala
    )

    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textura_id)
    glDrawArrays(GL_TRIANGLES, inicio, 6)


piso_inicio, piso_qtd, piso_tex = cria_piso("objetos/solo/piso.jpg", repeat=4)

15


In [764]:
def cria_terra(textura, repeat=1):
    global vertices_list, textures_coord_list

    size = 1.0  # tamanho grande

    # vértices (2 triângulos)
    terra = [
    (-size,-0.1,-size), ( size,-0.1, size), ( size,-0.1,-size),
    (-size,-0.1,-size), (-size,-0.1, size), ( size,-0.1, size),
    ]


    # UV padrão
    uv = [
        (0,0), (repeat,0), (repeat,repeat),
        (0,0), (repeat,repeat), (0,repeat),
    ]


    inicio = len(vertices_list)

    for v in terra:
        vertices_list.append(v)

    for t in uv:
        textures_coord_list.append(t)

    quantidade = len(terra)

    # carregar textura
    global numberTextures
    load_texture_from_file(numberTextures, textura)
    textura_id = numberTextures
    numberTextures += 1

    return inicio, quantidade, textura_id

def desenha_terra(inicio, textura_id, tx, ty, tz, sx=1, sy=1, sz=1):

    mat_model = model(
        0,          # angulo
        0,0,1,      # eixo rotação (irrelevante aqui)
        tx, ty, tz, # translação
        sx, sy, sz  # escala
    )

    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textura_id)
    glDrawArrays(GL_TRIANGLES, inicio, 6)


terra_inicio, terra_qtd, terra_tex = cria_terra("objetos/solo/terra.jpg", repeat=4)

16


In [765]:
def cria_grama(repeat = 1):
    global vertices_list, textures_coord_list

    size = 1.0  # tamanho grande

    # vértices (36 = 6 faces * 2 triângulos * 3 vértices)
    plane = [
    (-size,-0.1,-size), ( size,-0.1, size), ( size,-0.1,-size),
    (-size,-0.1,-size), (-size,-0.1, size), ( size,-0.1, size),
    ]

    # UV padrão
    uv = [
        (0,0), (repeat,0), (repeat,repeat),
        (0,0), (repeat,repeat), (0,repeat),
    ]

    inicio = len(vertices_list)

    for v in plane:
        vertices_list.append(v)

    for t in uv:
        textures_coord_list.append(t)

    quantidade = len(plane)

    # carregar textura
    global numberTextures
    load_texture_from_file(numberTextures, "objetos/solo/grama.jpg")
    textura_id = numberTextures
    numberTextures += 1

    return inicio, quantidade, textura_id

def desenha_grama(inicio, textura_id, tx, ty, tz, sx=1, sy=1, sz=1):

    mat_model = model(
        0,          # angulo
        0,0,1,      # eixo rotação (irrelevante aqui)
        tx, ty, tz, # translação
        sx, sy, sz  # escala
    )

    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textura_id)
    glDrawArrays(GL_TRIANGLES, inicio, 6)

grama_inicio, grama_qtd, grama_tex = cria_grama(repeat = 75) 

17


In [766]:
def cria_cubo(textura):
    global vertices_list, textures_coord_list

    size = 1.0

    # cubo completo (6 faces)
    cube = [
        # frente
        (-size,-size, size), ( size,-size, size), ( size, size, size),
        (-size,-size, size), ( size, size, size), (-size, size, size),

        # trás
        (-size,-size,-size), (-size, size,-size), ( size, size,-size),
        (-size,-size,-size), ( size, size,-size), ( size,-size,-size),

        # esquerda
        (-size,-size,-size), (-size,-size, size), (-size, size, size),
        (-size,-size,-size), (-size, size, size), (-size, size,-size),

        # direita
        ( size,-size,-size), ( size, size,-size), ( size, size, size),
        ( size,-size,-size), ( size, size, size), ( size,-size, size),

        # topo
        (-size, size,-size), (-size, size, size), ( size, size, size),
        (-size, size,-size), ( size, size, size), ( size, size,-size),

        # base
        (-size,-size,-size), ( size,-size,-size), ( size,-size, size),
        (-size,-size,-size), ( size,-size, size), (-size,-size, size),
    ]

    repeat = 2 

    uv = [
        (0,0), (repeat,0), (repeat,repeat),
        (0,0), (repeat,repeat), (0,repeat),
    ] * 6

    inicio = len(vertices_list)

    for v in cube:
        vertices_list.append(v)

    for t in uv:
        textures_coord_list.append(t)

    quantidade = len(cube)

    global numberTextures
    load_texture_from_file(numberTextures, textura)
    textura_id = numberTextures
    numberTextures += 1

    return inicio, quantidade, textura_id

def desenha_cubo(inicio, textura_id, tx, ty, tz, ang, rx, ry, rz, sx, sy, sz):

    mat_model = model(
        ang,
        rx, ry, rz,
        tx, ty, tz,
        sx, sy, sz
    )

    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glBindTexture(GL_TEXTURE_2D, textura_id)
    glDrawArrays(GL_TRIANGLES, inicio, 36)


cubo_inicio, cubo_qtd, cubo_tex = cria_cubo("objetos/parede/concreto.jpg")

def desenha_garagem():

    chao = -1.1

    largura = 2.0
    profundidade = 2.25
    altura = 1.0  # altura real da parede (total ≈ 3)

    # =========================
    # PAREDE DO FUNDO
    # =========================
    desenha_cubo(cubo_inicio, cubo_tex,
                 0, chao + altura, -profundidade,
                 0, 0,1,0,
                 largura, altura, 0.1)

    # =========================
    # PAREDE ESQUERDA
    # =========================
    desenha_cubo(cubo_inicio, cubo_tex,
                 -largura, chao + altura, 0,
                 0, 0,1,0,
                 0.1, altura, profundidade)

    # =========================
    # PAREDE DIREITA
    # =========================
    desenha_cubo(cubo_inicio, cubo_tex,
                 largura, chao + altura, 0,
                 0, 0,1,0,
                 0.1, altura, profundidade)

    # =========================
    # TETO
    # =========================
    desenha_cubo(cubo_inicio, cubo_tex,
                 0, chao + 2*altura, 0,
                 0, 0,1,0,
                 largura, 0.1, profundidade)


18


In [767]:
cubo_porta_inicio, cubo_porta_qtd, cubo_porta_tex = cria_cubo("objetos/porta/porta.jpg")

def desenha_porta():

    chao = -0.6

    largura = 1.9
    altura = 1.0
    espessura = 0.05

    frente = 2.3

    desenha_cubo(
        cubo_porta_inicio, cubo_porta_tex,
        0, chao + altura/2 + porta_offset, frente,
        0, 0,1,0,
        largura, altura, espessura
    )

19


### Requisitando buffer

In [768]:
buffer_VBO = glGenBuffers(2)

### Enviando vértices para a GPU

In [769]:
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

### Enviando textura para GPU

In [770]:
textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)]) # duas coordenadas
textures['position'] = textures_coord_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
stride = textures.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")

glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)


### Estado inicial da cena

In [771]:
# =====================
# ESTADO INICIAL
# =====================
INIT_cameraPos = glm.vec3(0.0, 0.0, 4.0)
INIT_cameraFront = glm.vec3(0.0, 0.0, -1.0)
INIT_yaw = -90.0
INIT_pitch = 0.0

INIT_car_x = 0.0
INIT_car_z = 0.0
INIT_car_angle = 0.0

INIT_pneu_scale = 0.05
INIT_porta_offset = 2.0


### Função de hitbox do carro

In [772]:
def carro_em_area_valida(x, z):

    # =====================
    # GARAGEM 
    # =====================
    dentro_garagem = (-0.65 <= x <= 0.65 and -0.8 <= z <= 3.5)

    if dentro_garagem:
        # só entra se a porta estiver aberta
        if porta_offset >= 1.25:
            return True
        else:
            return False

    # =====================
    # ESTRADA
    # =====================
    if -100.0 <= x <= 12.0 and 10.0 <= z <= 14:
        return True

    # =====================
    # TERRA 
    # =====================
    if -1.0 <= x <= 1.0 and 3.5 <= z <= 10.0:
        return True

    return False

def carro_debaixo_do_portao(x, z):
    # ajuste esses valores com base na posição real do seu portão
    return (-1.0 <= x <= 1.0) and (0.9 <= z <= 3.6)


### Funções do teclado

In [773]:
def reset_scene():
    global cameraPos, cameraFront, yaw, pitch
    global car_x, car_z, car_angle
    global pneu_scale, porta_offset

    cameraPos = glm.vec3(INIT_cameraPos)
    cameraFront = glm.vec3(INIT_cameraFront)
    yaw = INIT_yaw
    pitch = INIT_pitch

    car_x = INIT_car_x
    car_z = INIT_car_z
    car_angle = INIT_car_angle

    pneu_scale = INIT_pneu_scale
    porta_offset = INIT_porta_offset


def dentro_dos_limites(pos):
    limite_x = 10.0
    z_min = -2.5
    z_max = 17.5
    min_y = -1.0   # não atravessa o chão
    max_y = 20.0    # altura máxima 

    return (
        -limite_x <= pos.x <= limite_x and
        z_min <= pos.z <= z_max and
        min_y <= pos.y <= max_y
    )

def key_callback(window, key, scancode, action, mods):
    global porta_offset, car_x, car_z, car_angle, pneu_scale
    global vol_dir, rot, vel
    global cameraPos, cameraFront, cameraUp, polygonal_mode, deltaTime

    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(window, True)

    if action == glfw.PRESS or action == glfw.REPEAT:

        # =====================
        # PORTA
        # =====================
        if key == glfw.KEY_O:
            porta_offset = min(porta_offset + 0.05, 2.0)

        if key == glfw.KEY_L:
            if porta_offset > 0.0:  # ainda está aberto parcialmente
                if not carro_debaixo_do_portao(car_x, car_z):
                    porta_offset = max(porta_offset - 0.05, 0.0)



        # =====================
        # CARRO
        # =====================

        if key == glfw.KEY_UP:
            novo_angle = car_angle + vol_dir * rot

            novo_x = car_x + vel * math.sin(math.radians(novo_angle))
            novo_z = car_z + vel * math.cos(math.radians(novo_angle))

            if carro_em_area_valida(novo_x, novo_z):
                car_angle = novo_angle
                car_x = novo_x
                car_z = novo_z


        if key == glfw.KEY_DOWN:
            novo_angle = car_angle - vol_dir * rot

            novo_x = car_x - vel * math.sin(math.radians(novo_angle))
            novo_z = car_z - vel * math.cos(math.radians(novo_angle))

            if carro_em_area_valida(novo_x, novo_z):
                car_angle = novo_angle
                car_x = novo_x
                car_z = novo_z


        if key == glfw.KEY_LEFT:
            vol_dir = +1

        if key == glfw.KEY_RIGHT:
            vol_dir = -1

        if key == glfw.KEY_SPACE:
            vol_dir = 0

        # =====================
        # PNEU
        # =====================
        if key == glfw.KEY_I:
            pneu_scale += 0.05
            pneu_scale = min(pneu_scale, 0.45)

        if key == glfw.KEY_K:
            pneu_scale = max(0.05, pneu_scale - 0.05)

        # =====================
        # CÂMERA (WASD)
        # =====================
        cameraSpeed = 10.0 * deltaTime

        # W
        if key == glfw.KEY_W:
            nova_pos = cameraPos + cameraSpeed * cameraFront
            if dentro_dos_limites(nova_pos):
                cameraPos = nova_pos

        # S
        if key == glfw.KEY_S:
            nova_pos = cameraPos - cameraSpeed * cameraFront
            if dentro_dos_limites(nova_pos):
                cameraPos = nova_pos

        # A
        if key == glfw.KEY_A:
            direita = glm.normalize(glm.cross(cameraFront, cameraUp))
            nova_pos = cameraPos - direita * cameraSpeed
            if dentro_dos_limites(nova_pos):
                cameraPos = nova_pos

        # D
        if key == glfw.KEY_D:
            direita = glm.normalize(glm.cross(cameraFront, cameraUp))
            nova_pos = cameraPos + direita * cameraSpeed
            if dentro_dos_limites(nova_pos):
                cameraPos = nova_pos

        # reseta a cena (posição da câmera, do carro, etc)
        if key == glfw.KEY_R and action == glfw.PRESS:
            reset_scene()
            vol_dir = 0

        # =====================
        # MODO POLIGONAL
        # =====================
        if key == glfw.KEY_P and action == glfw.PRESS:
            polygonal_mode = not polygonal_mode


### Eventos da câmera

In [774]:
# camera
cameraPos   = glm.vec3(INIT_cameraPos)
cameraFront = glm.vec3(INIT_cameraFront)
cameraUp    = glm.vec3(0.0, 1.0, 0.0)

firstMouse = True
yaw   = INIT_yaw	# yaw is initialized to -90.0 degrees since a yaw of 0.0 results in a direction vector pointing to the right so we initially rotate a bit to the left.
pitch =  INIT_pitch
lastX =  largura / 2.0
lastY =  altura / 2.0
fov   =  45.0

# timing
deltaTime = 0.0	# time between current frame and last frame
lastFrame = 0.0


firstMouse = True
yaw = INIT_yaw
pitch = INIT_pitch
lastX =  largura/2
lastY =  altura/2

def framebuffer_size_callback(window, largura, altura):

    # make sure the viewport matches the new window dimensions note that width and 
    # height will be significantly larger than specified on retina displays.
    glViewport(0, 0, largura, altura)

# glfw: whenever the mouse moves, this callback is called
# -------------------------------------------------------
def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch
    
    if (firstMouse):

        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos # reversed since y-coordinates go from bottom to top
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1 # change this value to your liking
    xoffset *= sensitivity
    yoffset *= sensitivity

    yaw += xoffset
    pitch += yoffset

    # make sure that when pitch is out of bounds, screen doesn't get flipped
    if (pitch > 89.0):
        pitch = 89.0
    if (pitch < -89.0):
        pitch = -89.0

    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)

# glfw: whenever the mouse scroll wheel scrolls, this callback is called
# ----------------------------------------------------------------------
def scroll_callback(window, xoffset, yoffset):
    global fov

    fov -= yoffset
    if (fov < 1.0):
            fov = 1.0
    if (fov > 45.0):
            fov = 45.0
        
glfw.set_key_callback(window,key_callback)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

# tell GLFW to capture our mouse
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

### Matriz model, view e projection

In [775]:
def model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    
    angle = math.radians(angle)
    
    matrix_transform = glm.mat4(1.0) # instanciando uma matriz identidade
       
    # aplicando translacao (terceira operação a ser executada)
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))    
    
    # aplicando rotacao (segunda operação a ser executada)
    if angle!=0:
        matrix_transform = glm.rotate(matrix_transform, angle, glm.vec3(r_x, r_y, r_z))
    
    # aplicando escala (primeira operação a ser executada)
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))
    
    matrix_transform = np.array(matrix_transform)
    
    return matrix_transform

def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp);
    mat_view = np.array(mat_view)
    return mat_view

def projection():
    global altura, largura
    # perspective parameters: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 1000.0)

    
    mat_projection = np.array(mat_projection)    
    return mat_projection

def view_skybox():
    global cameraPos, cameraFront, cameraUp

    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp)

    # remove translação
    mat_view = glm.mat4(glm.mat3(mat_view))

    return np.array(mat_view)

### Exibindo janela

In [776]:
glfw.show_window(window)
glfw.set_key_callback(window, key_callback)

<function __main__.key_callback(window, key, scancode, action, mods)>

### Loop principal

In [777]:
glEnable(GL_DEPTH_TEST)   # importante para 3D
polygonal_mode = False

#=================================
# VARIÁVEIS GLOBAIS PARA ANIMAÇÃO
#=================================

# variáveis para animação
porta_offset = 2.0  # controla abertura (0 = fechada) (2.0 = totalmente aberta)

car_x = INIT_car_x
car_z = INIT_car_z
car_angle = INIT_car_angle

vol_dir = 0   # 1 = esquerda, 0 = reto, -1 = direita
vel = 0.1
rot = 2.0  # quanto gira por frame


pneu_scale = INIT_pneu_scale  # controla o estado do pneu (0 = murcho, 1 = cheio)
scale_full = INIT_porta_offset  # valor de pneu_scale para o pneu estar completamente cheio (ajuste para que fique visualmente adequado)

while not glfw.window_should_close(window):

    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    glfw.poll_events()

    glClearColor(1.0, 1.0, 1.0, 1.0)
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)

    # modo poligonal
    if polygonal_mode:
        glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)

    # ==========================================================
    # PROJECTION 
    # ==========================================================

    mat_projection = projection()
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)

    # ==========================================================
    # SKYBOX 
    # ==========================================================

    mat_view = view_skybox()
    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_model = np.identity(4)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    glDepthFunc(GL_LEQUAL)
    glDepthMask(GL_FALSE)

    desenha_skybox(skybox_inicio, skybox_tex)

    glDepthMask(GL_TRUE)
    glDepthFunc(GL_LESS)

    # ==========================================================
    # OBJETOS NORMAIS 
    # ==========================================================
        
    mat_view = view()
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    # desenha a garagem e suas portas (ambiente interno)
    desenha_garagem()
    desenha_porta()
        
    # desenha os solos do cenário (ambiente interno e externo)
    desenha_grama(grama_inicio, grama_tex, 0, -1, 0, 100, 1, 100)
    desenha_piso(piso_inicio, piso_tex, 0, -0.99, 0, 2.0, 1, 2.25)
    for i in range(3):
        desenha_terra(terra_inicio, terra_tex, 0, -0.995, 3 + i * 4.0, 1.5, 1, 2.0)


    # desenha a estrada (ambiente externo)
    for i in range(8):
        desenha_rua(90, 0, 1, 0, 0 + i * 13.15, -1.12, 12.5, 0.4, 0.4, 0.4, 0)
        desenha_rua(90, 0, 1, 0, 0 - i * 13.15, -1.12, 12.5, 0.4, 0.4, 0.4, 0)

    # desenha os postes de iluminação (ambiente externo)
    for i in range(3):
        desenha_poste(90, 0, 1, 0, 5 + i * 30, -1.1, 15, 0.5, 0.5, 0.5, 1)
        desenha_poste(90, 0, 1, 0, -25 - i * 30, -1.1, 15, 0.5, 0.5, 0.5, 1)

    # desenha veículos (ambiente interno e externo) (translada)
    desenha_carro(car_angle, 0, 1, 0, car_x, -0.75, car_z, 0.8, 0.8, 0.8, 2)

    desenha_carro(180, 0, 0, 1, 17.0, -0.70, 12.5, 0.8, 0.8, 0.8, 2)

    # desenha conjunto de pneus (ambiente interno) (escala)
    t = pneu_scale / scale_full
    t = max(0.0, min(1.0, t))

    # escala
    scale_y = 0.30 + 0.15 * t
    scale_xz = 0.50 - 0.05 * t

    # posição corrigida
    chao_y = -1.13
    scale_y_min = 0.30

    fator_subida = 0.4
    pneu_y = chao_y + (scale_y_min / 2.0) + (scale_y - scale_y_min)  * fator_subida
 
    desenha_pneu(90, 0, 1, 0, -1.8, pneu_y, 0.5, scale_xz, scale_y, scale_xz, 3)

    # desenha cones de sinalização (ambiente externo)
    for i in range(5):
        desenha_cone(0.0, 0, 0, 1, 15.0, -1.0, 11.5 + i*0.5, 0.5, 0.5, 0.5, 4)

    # desenha conjunto de ferramentas, tanques de gasolina e lâmpada (ambiente interno)
    desenha_ferramentas(0.0, 0, 1, 0, -1.2, -1.09, -1.95, 0.1, 0.1, 0.1, 5)
    desenha_gasolina(180, 0, 1, 0, 2.0, -0.95, -2.1, 0.08, 0.08, 0.08, 6)
    desenha_lampada(0.0, 0, 1, 0, 0, 0.23, 0, 0.5, 0.5, 0.5, 7)

    # desenha conjunto de árvores do lado oposto da via (ambiente externo) (preenche o cenário e dá sensação de profundidade)
    for i in range(20):
        desenha_arvore(0.0, 0, 0, 1, 0 + i * 5, -0.975, 17.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -5 - i * 5, -0.975, 17.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 2.5 + i * 5, -0.975, 22.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -2.5 - i * 5, -0.975, 22.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 0 + i * 5, -0.975, 27.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -5 - i * 5, -0.975, 27.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 2.5 + i * 5, -0.975, 32.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -2.5 - i * 5, -0.975, 32.5, 0.4, 0.4, 0.4, 8)

    # desenha conjunto de árvores na via do lado da garagem (ambiente externo) (preenche o cenário e dá sensação de profundidade)
    for i in range(20):
        desenha_arvore(0.0, 0, 0, 1, 5.0 + i * 5, -0.975, 7.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -5.0 - i * 5, -0.975, 7.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 5 + i * 5, -0.975, 2.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -5 - i * 5, -0.975, 2.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 5.0 + i * 5, -0.975, -2.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -5.0 - i * 5, -0.975, -2.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 0 + i * 5, -0.975, -7.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -5 - i * 5, -0.975, -7.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -2.5 + i * 5, -0.975, -12.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -7.5 - i * 5, -0.975, -12.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, 0 + i * 5, -0.975, -17.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -5 - i * 5, -0.975, -17.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -2.5 + i * 5, -0.975, -22.5, 0.4, 0.4, 0.4, 8)
        desenha_arvore(0.0, 0, 0, 1, -7.5 - i * 5, -0.975, -22.5, 0.4, 0.4, 0.4, 8)
    

    glfw.swap_buffers(window)

glfw.terminate()
